# RAG Pipelines - Data ingestion to vector DB Pipeline


In [1]:
print("Cell 2")

# Standard Library
import os
import uuid
from typing import List, Dict, Any

# Numerical Computing
import numpy as np

# LangChain Loaders
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader,
    TextLoader
)

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_community.embeddings import OpenAIEmbeddings

# Vector Database
import chromadb
from chromadb.config import Settings

Cell 2


/tmp/ipykernel_231637/2286867029.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/moeen/projects/AgenticAI/.venv/lib/python3.12/site-packages/ipykernel_launcher

In [2]:
print("Cell 3")

import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


def load_pdf_documents(directory_path: str):
    """
    Load all PDF files from a directory.
    """

    loader = DirectoryLoader(
        path=directory_path,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True
    )

    documents = loader.load()

    print(f"Loaded {len(documents)} pages from PDF files.")

    return documents


def add_source_metadata(documents):
    """
    Add filename to the metadata of each document.
    """

    for doc in documents:
        doc.metadata["filename"] = os.path.basename(
            doc.metadata["source"]
        )

    return documents


def display_documents(documents, preview_length=200):
    """
    Display document metadata and content preview.
    """

    print(f"\nTotal Documents: {len(documents)}\n")

    for i, doc in enumerate(documents, start=1):

        print(f"Document/Page {i}")

        print("Metadata:")
        print(doc.metadata)

        print("\nPreview:")
        print(doc.page_content[:preview_length])

        print("=" * 80)


# ---------------- MAIN ---------------- #

pdf_directory = "/home/moeen/projects/AgenticAI/pdf"
print(f"Using PDF directory: {pdf_directory}")

documents = load_pdf_documents(pdf_directory)

documents = add_source_metadata(documents)

display_documents(documents)

Cell 3
Using PDF directory: /home/moeen/projects/AgenticAI/pdf


100%|██████████| 3/3 [00:06<00:00,  2.24s/it]

Loaded 527 pages from PDF files.

Total Documents: 527

Document/Page 1
Metadata:
{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/home/moeen/projects/AgenticAI/pdf/MACHINE LEARNING(R17A0534).pdf', 'total_pages': 120, 'page': 0, 'page_label': '1', 'filename': 'MACHINE LEARNING(R17A0534).pdf'}

Preview:
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(
Document/Page 2
Metadata:
{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/home/moeen/projects/AgenticAI/pdf/MACHINE LEARNING(R17A0534).pdf', 'total_pages': 120, 'page': 1, 'page_label': '2', 'filename': 'MACHINE LEARNING(R17A0534).pdf'}

Preview:
IV Year B. Tech. CSE –II Sem                        L   T/P/D   C  
  4   1/- / -   3  
(R17A0534) Machine Learning 
Objectives:  
 Acquire theoretical Knowledge on setting hypothesis for pattern 

In [3]:
print("Cell 4")

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into smaller chunks for embedding.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Total Chunks Created: {len(split_docs)}")
    print(f"Split {len(documents)} document pages into {len(split_docs)} chunks.\n")

    # Display an example chunk
    if split_docs:
        print("Example Chunk")
        print("-" * 50)
        print(split_docs[0].page_content[:200] + "...")
        print("\nMetadata:")
        print(split_docs[0].metadata)
        print("-" * 50)

    return split_docs

Cell 4


In [4]:
print("Cell 5")

# Split the loaded documents into chunks
chunks = split_documents(documents)

# Display basic information
print(f"\nTotal Chunks: {len(chunks)}")

# Display the first chunk
print("\nFirst Chunk:")
print("-" * 50)
print(chunks[0].page_content[:300] + "...")

print("\nMetadata:")
print(chunks[0].metadata)

Cell 5
Total Chunks Created: 1362
Split 527 document pages into 1362 chunks.

Example Chunk
--------------------------------------------------
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(...

Metadata:
{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/home/moeen/projects/AgenticAI/pdf/MACHINE LEARNING(R17A0534).pdf', 'total_pages': 120, 'page': 0, 'page_label': '1', 'filename': 'MACHINE LEARNING(R17A0534).pdf'}
--------------------------------------------------

Total Chunks: 1362

First Chunk:
--------------------------------------------------
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) 

# Embedding and Vector DB


In [5]:
print("Cell 5")

# Split the loaded documents into chunks
chunks = split_documents(documents)

# Display basic information
print(f"\nTotal Chunks: {len(chunks)}")

# Display the first chunk
print("\nFirst Chunk:")
print("-" * 50)
print(chunks[0].page_content[:300] + "...")

print("\nMetadata:")
print(chunks[0].metadata)

Cell 5
Total Chunks Created: 1362
Split 527 document pages into 1362 chunks.

Example Chunk
--------------------------------------------------
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(...

Metadata:
{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/home/moeen/projects/AgenticAI/pdf/MACHINE LEARNING(R17A0534).pdf', 'total_pages': 120, 'page': 0, 'page_label': '1', 'filename': 'MACHINE LEARNING(R17A0534).pdf'}
--------------------------------------------------

Total Chunks: 1362

First Chunk:
--------------------------------------------------
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) 

In [6]:
print("Cell 7")

import uuid
import numpy as np
import re
from typing import List, Dict, Any


class EmbeddingManager:
    """
    Handles embedding generation using OpenAI embeddings.
    """

    def __init__(self, model_name: str = "text-embedding-3-small"):
        self.model_name = model_name
        self.model = None
        self.use_fallback = False
        self.vocabulary = None

    def load_model(self):
        """
        Initialize the embedding model.
        """
        try:
            self.model = OpenAIEmbeddings(model=self.model_name)
            print("OpenAI embeddings initialized successfully!")

        except Exception as e:
            print(f"Error initializing OpenAI embeddings: {e}")
            print("Falling back to a lightweight local embedding function.")
            self.model = None
            self.use_fallback = True

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        """
        Convert list of texts into embeddings.
        """
        if self.model is None:
            if not self.use_fallback:
                raise ValueError("Model not loaded. Call load_model() first.")
            return self._simple_embed(texts)

        if self.use_fallback:
            return self._simple_embed(texts)
        return self.model.embed_documents(texts)

    def _simple_embed(self, texts: List[str]) -> List[List[float]]:
        """Generate a simple local embedding when OpenAI embeddings are unavailable."""
        first_time = self.vocabulary is None
        token_counts = []
        vocabulary = set()

        for text in texts:
            tokens = re.findall(r"\w+", text.lower())
            counts = {}
            for token in tokens:
                counts[token] = counts.get(token, 0) + 1
            token_counts.append(counts)
            vocabulary.update(counts.keys())

        if first_time:
            self.vocabulary = sorted(vocabulary)[:256]

        embedded_texts = []
        for counts in token_counts:
            vector = [float(counts.get(token, 0)) for token in self.vocabulary]
            norm = np.linalg.norm(vector)
            if norm > 0:
                vector = [value / norm for value in vector]
            embedded_texts.append(vector)

        return embedded_texts
    def create_embedding_dict(self, chunks: List[Any]) -> List[Dict[str, Any]]:
        """
        Convert Document chunks into embedding-ready dictionaries.
        """

        texts = [chunk.page_content for chunk in chunks]
        embeddings = self.embed_texts(texts)

        results = []

        for i, chunk in enumerate(chunks):
            embedding = embeddings[i]
            if hasattr(embedding, "tolist"):
                embedding = embedding.tolist()

            results.append({
                "id": str(uuid.uuid4()),
                "content": chunk.page_content,
                "metadata": chunk.metadata,
                "embedding": embedding
            })

        return results

    def generate_embeddings(self, chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """
        Full pipeline: chunks → embeddings + metadata structure
        """

        print(f"Generating embeddings for {len(chunks)} chunks...")

        embedded_data = self.create_embedding_dict(chunks)

        print("Embeddings generated successfully!")

        return embedded_data

Cell 7


In [7]:
print("Cell 8")

import chromadb
from chromadb.config import Settings
import uuid
import numpy as np
from typing import List, Dict, Any


class VectorStore:
    """
    Handles storage and retrieval of embeddings using ChromaDB.
    """

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "./vector_store"
    ):

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize ChromaDB client and collection.
        """

        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name
        )

        print(f"Vector store initialized: {self.collection_name}")

    def add_documents(
        self,
        documents: List[Dict[str, Any]],
        embeddings: List[List[float]]
    ):
        """
        Add documents + embeddings to vector DB.
        """

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings must have same length")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        contents = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc["metadata"])
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc["content"])
            metadatas.append(metadata)

            # Content
            contents.append(doc["content"])

            # Embedding
            if hasattr(embedding, "tolist"):
                embeddings_list.append(embedding.tolist())
            else:
                embeddings_list.append(list(embedding))

        # Add to ChromaDB
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=contents
        )

        print(f"Successfully added {len(documents)} documents to vector store")

Cell 8


In [8]:
print("Cell 9")

# Just to verify chunks exist and are structured correctly
print(f"Total chunks available: {len(chunks)}\n")

# Show first chunk safely
if chunks:
    print("Example Chunk:")
    print("-" * 50)
    print("Content:")
    print(chunks[0].page_content[:300])

    print("\nMetadata:")
    print(chunks[0].metadata)
    print("-" * 50)

Cell 9
Total chunks available: 1362

Example Chunk:
--------------------------------------------------
Content:
MACHINE LEARNING  
[R17A0534] 
LECTURE NOTES 
 
B.TECH IV YEAR – I SEM(R17) 
(2020-21) 
 
 
 
 
 
 
DEPARTMENT OF 
COMPUTER SCIENCE AND ENGINEERING 
MALLA REDDY COLLEGE OF ENGINEERING & 
TECHNOLOGY 
(Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Af

Metadata:
{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/home/moeen/projects/AgenticAI/pdf/MACHINE LEARNING(R17A0534).pdf', 'total_pages': 120, 'page': 0, 'page_label': '1', 'filename': 'MACHINE LEARNING(R17A0534).pdf'}
--------------------------------------------------


In [9]:
print("Cell 10")

# Step 1: Extract text from chunks (for embedding model)
texts = [chunk.page_content for chunk in chunks]

print(f"Generating embeddings for {len(texts)} chunks...")

# Step 2: Initialize embedding manager and generate embeddings
embedding_manager = EmbeddingManager()
embedding_manager.load_model()
embedded_chunks = embedding_manager.generate_embeddings(chunks)

print(f"Generated {len(embedded_chunks)} embedded chunks")

# Step 3: Initialize vector store
vector_store = VectorStore(
    collection_name="pdf_documents",
    persist_directory="./vector_store"
)

# Step 4: Add embeddings to vector database
vector_store.add_documents(
    documents=embedded_chunks,
    embeddings=[item["embedding"] for item in embedded_chunks]
)

print("All documents successfully stored in vector database!")

/tmp/ipykernel_231637/3923458152.py:24: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  self.model = OpenAIEmbeddings(model=self.model_name)


Cell 10
Generating embeddings for 1362 chunks...
Error initializing OpenAI embeddings: 1 validation error for OpenAIEmbeddings
  Value error, Did not find openai_api_key, please add an environment variable `OPENAI_API_KEY` which contains it, or pass `openai_api_key` as a named parameter. [type=value_error, input_value={'model': 'text-embedding...20, 'http_client': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
Falling back to a lightweight local embedding function.
Generating embeddings for 1362 chunks...
Embeddings generated successfully!
Generated 1362 embedded chunks
Vector store initialized: pdf_documents
Adding 1362 documents to vector store...
Successfully added 1362 documents to vector store
All documents successfully stored in vector database!


## Retriever Pipeline from VectoreStore


In [10]:
print(  "Cell 11")

class RAGRetriever:
    """
    handle query - based retrieval from the vecoter store 
    """

    def __init__(self, vector_store: VectorStore,embedding_manger: EmbeddingManager):
        """
        Initialize the RAGRetriever with a vector store and embedding manager.

        Args:
            vector_store (VectorStore): The vector store instance for storing and retrieving embeddings.
            embedding_manager (EmbeddingManager): The embedding manager instance for generating embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int = 5,score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        """
        Retrieve relevant documents based on the query.

        Args:
            query (str): The input query string.
            top_k (int): The number of top results to retrieve.
            score_threshold (float): Minimum score threshold for filtering results.

        Returns:
            List[Dict[str, Any]]: A list of retrieved documents with metadata and scores.
        """

        # Step 1: Generate embedding for the query
        query_embedding = self.embedding_manager.embed_texts([query])[0]

        #search in vecoter store and return the results
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k,
                include=["metadatas", "documents", "distances"]
            )

            retrieved_docs: List[Dict[str, Any]] = []

            if results and results.get("ids") and results["ids"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for idx, (doc_id, doc_content, doc_metadata, doc_distance) in enumerate(
                    zip(ids, documents, metadatas, distances), start=1
                ):
                    similarity_score = 1 - doc_distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": doc_content,
                            "metadata": doc_metadata,
                            "score": similarity_score,
                            "similarity_score": similarity_score,
                            "distance": doc_distance,
                            "rank": idx,
                        })

            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")
            else:
                print(f"No documents retrieved for query: '{query}'")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manger=embedding_manager)



Cell 11


In [11]:
print("Cell 12")
rag_retriever

Cell 12


In [12]:
print("cell 13")
results = rag_retriever.retrieve("what is machine learning", top_k=3)
if results:
    for r in results:
        print(f"  Score: {r['score']:.3f} | {r['content'][:100]}...")
else:
    print("No results retrieved. Check the vector store.")


cell 13
Error during retrieval: Collection expecting embedding with dimension of 256, got 7


[]

## Integration VectorDB Context popeline with LLM output


In [13]:
print("cell 14")
#Simple RAGPipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

### initialize the groq llm model
groq_api_key = os.getenv("GROQ_API_KEY", "<your_api_key_here>")

llm = ChatGroq(api_key=groq_api_key, model="llama-3.3-70b-versatile", temperature=0.7, max_tokens=500)

##2 simple RAG function retriever context + generate response
def rag_query(query:str,rag_retriever:RAGRetriever,llm:ChatGroq,top_k:int=5,score_threshold:float=0.0)->str:
    """
    Perform a RAG query by retrieving relevant documents and generating a response using the LLM.

    Args:
        query (str): The input query string.
        rag_retriever (RAGRetriever): The RAG retriever instance for document retrieval.
        llm (ChatGroq): The Groq LLM instance for response generation.
        top_k (int): The number of top results to retrieve.
        score_threshold (float): Minimum score threshold for filtering results.

    Returns:
        str: The generated response from the LLM based on retrieved documents.
    """

    # Step 1: Retrieve relevant documents
    retrieved_docs = rag_retriever.retrieve(query=query, top_k=top_k, score_threshold=score_threshold)

    # Step 2: Prepare context for the LLM
    context = "\n\n".join([f"Document {i+1}:\n{doc['content']}" for i, doc in enumerate(retrieved_docs)])

    # Step 3: Generate response using the LLM
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt).content

    return response
    


cell 14


In [14]:
print("cell 14")

print("=" * 70)
print("  BASIC RAG QUERY")
print("=" * 70)

question = "what is machine learning"
print(f"Question: {question}\n")

answer = rag_query(question, rag_retriever, llm)

print("-" * 70)
print("ANSWER:")
print("-" * 70)
print(answer)
print("-" * 70)


cell 14
Error during retrieval: Collection expecting embedding with dimension of 256, got 7


AttributeError: 'str' object has no attribute 'content'

## Enhanced RAG Pipeline Features


In [ ]:
# Enhanced RAG pipeline Feature 
print("cell 15")

def rag_advanced(query,retriever,llm,top_k:int=5,min_score=0.2,return_context=False):
    """
    Perform an advanced RAG query with additional features.

    Args:
        query (str): The input query string.
        retriever (RAGRetriever): The RAG retriever instance for document retrieval.
        llm (ChatGroq): The Groq LLM instance for response generation.
        top_k (int): The number of top results to retrieve.
        score_threshold (float): Minimum score threshold for filtering results.

    Returns:
        str: The generated response from the LLM based on retrieved documents.
    """
    results=retriever.retrieve(query=query,top_k=top_k,score_threshold=min_score)
    #prepare context sources
    context="\n\n".join([f"Document {i+1}:\n{doc['content']}" for i, doc in enumerate(results)])

    sources = []
    for doc in results:
        source_info = doc["metadata"].get("source_file") or doc["metadata"].get("source", "unknown")
        if isinstance(source_info, dict):
            source_value = source_info.get("filename", str(source_info))
        else:
            source_value = str(source_info)

        sources.append({
            "source": source_value,
            "page": doc["metadata"].get("page_number", doc["metadata"].get("page", "unknown")),
            "score": doc["score"],
            "preview": doc["content"][:200] + "..."
        })
    confidence = max([doc["score"] for doc in results], default=0.0)

    #generaett answer from llm

    prompt = f"use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt).content

    output={
        "query":query,
        "response":response,
        "confidence":confidence,
        "sources":sources
    }
    return output


## Example usage
print("=" * 70)
print("  ADVANCED RAG QUERY")
print("=" * 70)

question = "what is machine learning"
print(f"Question: {question}\n")

result = rag_advanced(question, rag_retriever, llm, top_k=5, min_score=0.2, return_context=True)

print("-" * 70)
print("ANSWER:")
print("-" * 70)
print(result["response"])
print()
print("-" * 70)
print(f"CONFIDENCE: {result['confidence']:.3f}")
print("-" * 70)
print()
print("SOURCES:")
print("-" * 70)
for src in result["sources"]:
    print(f"  File : {src['source']}")
    print(f"  Page : {src['page']}")
    print(f"  Score: {src['score']:.4f}")
    print(f"  Preview: {src['preview']}")
    print("-" * 70)


In [ ]:
print("cell 16")

#Advanced RAG pipeline Streaming, Citations, History, Summarization

from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    """
    An advanced RAG pipeline that supports streaming, citations, history, and summarization.
    """

    def __init__(self, retriever: RAGRetriever, llm: ChatGroq):
        self.retriever = retriever
        self.llm = llm
        self.history: List[Dict[str, Any]] = []

    def query(self, query: str, top_k: int = 5, min_score: float = 0.2) -> Dict[str, Any]:
        """
        Perform a query using the advanced RAG pipeline.

        Args:
            query (str): The input query string.
            top_k (int): The number of top results to retrieve.
            min_score (float): Minimum score threshold for filtering results.

        Returns:
            Dict[str, Any]: A dictionary containing the response, confidence score, sources, and context.
        """
        # Retrieve relevant documents
        results = self.retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)

        # Prepare context for the LLM
        context = "\n\n".join([f"Document {i+1}:\n{doc['content']}" for i, doc in enumerate(results)])

        # Generate response using the LLM
        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        response = self.llm.invoke(prompt).content

        # Prepare sources information
        sources = []
        for doc in results:
            source_info = doc["metadata"].get("source_file") or doc["metadata"].get("source", "unknown")
            if isinstance(source_info, dict):
                source_value = source_info.get("filename", str(source_info))
            else:
                source_value = str(source_info)

            sources.append({
                "source": source_value,
                "page": doc["metadata"].get("page_number", doc["metadata"].get("page", "unknown")),
                "score": doc["score"],
                "preview": doc["content"][:200] + "..."
            })

        confidence = max([doc["score"] for doc in results], default=0.0)

        # Store in history
        self.history.append({
            "query": query,
            "response": response,
            "confidence": confidence,
            "sources": sources,
            "timestamp": time.time()
        })

        return {
            "query": query,
            "response": response,
            "confidence": confidence,
            "sources": sources,
            "context": context
        }


## Example usage
print("=" * 70)
print("  ADVANCED RAG PIPELINE (Class)")
print("=" * 70)

pipeline = AdvancedRAGPipeline(rag_retriever, llm)
question = "what is machine learning"
print(f"Question: {question}\n")

result = pipeline.query(question, top_k=5, min_score=0.2)

print("-" * 70)
print("ANSWER:")
print("-" * 70)
print(result["response"])
print()
print("-" * 70)
print(f"CONFIDENCE: {result['confidence']:.3f}")
print("-" * 70)
print()
print("SOURCES:")
print("-" * 70)
for src in result["sources"]:
    print(f"  File : {src['source']}")
    print(f"  Page : {src['page']}")
    print(f"  Score: {src['score']:.4f}")
    print(f"  Preview: {src['preview']}")
    print("-" * 70)
print()
print("HISTORY:")
print("-" * 70)
for entry in pipeline.history:
    print(f"  Q: {entry['query']}")
    print(f"  A: {entry['response'][:100]}...")
    print(f"  At: {time.ctime(entry['timestamp'])}")
    print("-" * 70)


In [ ]:
print("=" * 70)
print("  RAG PIPELINE COMPARISON")
print("  Same question tested on all 3 pipelines")
print("=" * 70)

question = "what is reinforcement learning"
print("\nQuestion: " + question + "\n")
print("=" * 70)

# --- Pipeline 1: Basic RAG ---
print("\n[1] BASIC RAG (rag_query)")
print("    Returns just the answer string.")
print("-" * 50)
answer = rag_query(question, rag_retriever, llm)
print(answer)

# --- Pipeline 2: Advanced RAG ---
print("\n" + "=" * 70)
print("[2] ADVANCED RAG (rag_advanced)")
print("    Returns answer + confidence + sources.")
print("-" * 50)
result = rag_advanced(question, rag_retriever, llm, top_k=3, min_score=0.2)
print("Answer: " + str(result["response"]))
print("\nConfidence: " + str(round(result["confidence"], 3)))
print("\nSources:")
for s in result["sources"]:
    print("  - " + s["source"] + " (page " + str(s["page"]) + ") [score: " + str(round(s["score"], 3)) + "]")

# --- Pipeline 3: AdvancedRAGPipeline Class ---
print("\n" + "=" * 70)
print("[3] ADVANCED RAG PIPELINE (Class)")
print("    Returns answer + confidence + sources + history tracking.")
print("-" * 50)
pipeline2 = AdvancedRAGPipeline(rag_retriever, llm)
result2 = pipeline2.query(question, top_k=3, min_score=0.2)
print("Answer: " + str(result2["response"]))
print("\nConfidence: " + str(round(result2["confidence"], 3)))
print("\nSources:")
for s in result2["sources"]:
    print("  - " + s["source"] + " (page " + str(s["page"]) + ") [score: " + str(round(s["score"], 3)) + "]")
print("\nHistory entries: " + str(len(pipeline2.history)))

print("\n" + "=" * 70)
print("  COMPARISON COMPLETE")
print("=" * 70)


## Difference Between the Three Pipelines

| Feature | Basic `rag_query` | `rag_advanced` | `AdvancedRAGPipeline` |
|---------|-------------------|----------------|------------------------|
| **Return type** | Plain string | Dictionary | Dictionary |
| **Answer** | Yes | Yes | Yes |
| **Confidence score** | No | Yes | Yes |
| **Source citations** | No | Yes (file, page, score, preview) | Yes (file, page, score, preview) |
| **Context text** | No | Optional (`return_context=True`) | Always included |
| **Conversation history** | No | No | Yes (with timestamps) |
| **Interface** | Function | Function | Class (reusable instance) |

### Summary
- **Basic** — minimal, good for quick prototyping. Returns just the answer string.
- **Advanced** — adds source transparency and confidence scoring. Good for production where users need to see where the answer came from.
- **Pipeline class** — adds history tracking with timestamps. Ideal for chat applications that need conversation context.
